# 3주차 직접 해보기: 신경망, 역전파, 언어 모델

사이트의 3주차 회독과 정리 슬라이드를 본 다음 풀어요. 위에서부터 차례로 `Shift+Enter` 로 실행해요.

- `# TODO` 칸의 `None` 을 알맞은 코드나 값으로 바꾸고, 바로 아래 **확인 셀**을 실행해요.
- 막히면 맨 아래 **정답 코드**를 봐요.

Colab 에는 PyTorch 가 이미 깔려 있어요. GPU 는 필요 없어요.

In [ ]:
import math
import torch
def ok(cond, msg):
    assert cond, msg
    print('정답! ' + msg)
print('PyTorch', torch.__version__)

## 1. 뉴런 하나 계산하기

강의 N3 p.9-10, 기초 다지기 9단원. 뉴런 = **가중합(입력 x 가중치를 모두 더함) + 편향 항(bias term)** 다음에 **활성화 함수**. ReLU 는 음수면 0, 양수면 그대로예요.

In [ ]:
x = [1.0, 2.0, -1.0]
w = [0.5, -0.25, 1.0]
b = 0.5

z = sum(xi * wi for xi, wi in zip(x, w)) + b
h = None   # TODO: ReLU(z) = max(0, z)
print(z, h)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(z - (-0.5)) < 1e-9, '0.5 - 0.5 - 1.0 + 0.5 = -0.5')
ok(h == 0.0, '음수라서 ReLU 를 지나면 0')

## 2. 비선형이 없으면 층을 쌓아도 한 층

강의 N3 p.11. 활성화 함수 없이 `W2 @ (W1 @ x)` 를 계산하면 `(W2 @ W1) @ x`, 곧 행렬 하나를 곱한 것과 같아요. `@` 는 행렬 곱이에요.

In [ ]:
torch.manual_seed(0)
W1 = torch.randn(4, 3)
W2 = torch.randn(2, 4)
x = torch.randn(3)

two_layers = W2 @ (W1 @ x)
W = None   # TODO: 두 행렬을 미리 곱한 한 행렬
one_layer = W @ x
print(two_layers, one_layer, W.shape)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(torch.allclose(two_layers, one_layer, atol=1e-5), '두 층 = 한 층')
ok(tuple(W.shape) == (2, 3), '(2,4) @ (4,3) = (2,3)')

## 3. shape 먼저 예측하기

강의 N3 p.25-26, 실습 N3L p.3. `(행, 열) @ (열,)` 은 `(행,)` 이 돼요. 앞 행렬의 열 수와 뒤 벡터 길이가 같아야 곱할 수 있어요. 코드를 돌리기 **전에** 튜플로 답을 적어요.

In [ ]:
x = torch.randn(5)
W = torch.randn(3, 5)
b = torch.randn(3)
u = torch.randn(3)

z = W @ x + b
s = u @ torch.tanh(z)

my_z_shape = None   # TODO: 예: (7,)
my_s_shape = None   # TODO: 숫자 하나(스칼라)는 ()

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(tuple(z.shape) == my_z_shape, 'z 는 (3,)')
ok(tuple(s.shape) == my_s_shape, 's 는 스칼라 ()')

## 4. 손으로 미분한 값과 autograd 비교

강의 N3 p.18, p.32, 실습 N3L p.4. f(x) = 3x² + 5x 이면 f'(x) = 6x + 5 예요. `requires_grad=True` 로 만든 텐서에 `backward()` 를 부르면 PyTorch 가 `x.grad` 에 기울기를 넣어 줘요.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
f = 3 * x**2 + 5 * x
f.backward()

by_hand = None   # TODO: x=1 에서 6x + 5 의 값
print(x.grad.item(), by_hand)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(x.grad.item() - by_hand) < 1e-6 and by_hand == 11, 'autograd 와 손계산 모두 11')

## 5. 연쇄 법칙

강의 N3 p.20-24. f = (2x + 1)² 를 u = 2x + 1, f = u² 로 나누면 df/dx = (df/du) x (du/dx) = 2u x 2 예요.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
u = 2 * x + 1
f = u ** 2
f.backward()

df_du = None   # TODO: 2u 의 값 (u = 3)
du_dx = None   # TODO: 2x + 1 을 x 로 미분한 값
print(x.grad.item(), df_du * du_dx)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(df_du * du_dx == 12 and abs(x.grad.item() - 12) < 1e-6, '6 x 2 = 12')

## 6. 계산 그래프 노드 직관: +, max, x

강의 N3 p.30. 위에서 내려온 기울기가 1일 때,

- **+** 는 기울기를 두 입력에 그대로 나눠 줘요.
- **max** 는 더 큰 입력 쪽에만 보내요(작은 쪽은 0).
- **x** 는 서로 상대편 값을 곱해 줘요.

f = (x + y) x max(z, w), x=1, y=2, z=3, w=-1 에서 네 기울기를 손으로 적어요.

In [ ]:
grads_by_hand = {'x': None, 'y': None, 'z': None, 'w': None}   # TODO

x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
z = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(-1.0, requires_grad=True)
f = (x + y) * torch.maximum(z, w)
f.backward()
auto = {'x': x.grad.item(), 'y': y.grad.item(), 'z': z.grad.item(), 'w': w.grad.item()}
print(auto)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(all(abs(auto[k] - grads_by_hand[k]) < 1e-6 for k in auto), 'x, y 는 max(z,w)=3, z 는 x+y=3, w 는 작은 쪽이라 0')

## 7. 바이그램(2-gram) 언어 모델: 세어서 확률 구하기

강의 N3 p.37-38. P(다음 단어 | 앞 단어) = count(앞 단어, 다음 단어) / count(앞 단어). 문장 앞뒤에 `<s>`, `</s>` 를 붙였어요.

In [ ]:
from collections import Counter
sents = [['<s>', '나는', '영화', '를', '봤다', '</s>'],
         ['<s>', '나는', '책', '을', '봤다', '</s>'],
         ['<s>', '너는', '영화', '를', '봤다', '</s>']]

uni = Counter(w for s in sents for w in s[:-1])
bi = Counter((s[i], s[i + 1]) for s in sents for i in range(len(s) - 1))

def p(nxt, prev):
    return None   # TODO: bi[(prev, nxt)] / uni[prev]

print(p('나는', '<s>'), p('영화', '나는'))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(p('나는', '<s>') - 2 / 3) < 1e-9, '<s> 3번 중 나는 2번')
ok(abs(p('영화', '나는') - 0.5) < 1e-9, '나는 2번 중 영화 1번')
ok(p('책', '너는') == 0, '한 번도 안 센 쌍은 0: 희소성 문제(N3 p.39)')

## 8. 퍼플렉서티(Perplexity)

강의 N3 p.41-42. 모델이 정답 토큰에 준 확률이 p1, ..., pN 이면

PPL = exp( -(1/N) x (log p1 + ... + log pN) ) = (p1 x ... x pN)^(-1/N)

낮을수록 좋아요. 모든 토큰에 확률 1/4 를 주면 PPL 은 4, 곧 '평균 4개 후보 사이에서 헷갈리는 정도'예요.

In [ ]:
probs = [0.5, 0.25, 0.125, 0.5]

mean_nll = -sum(math.log(q) for q in probs) / len(probs)
ppl = None   # TODO: math.exp 사용

print(round(mean_nll, 4), round(ppl, 4))

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(abs(ppl - (0.5 * 0.25 * 0.125 * 0.5) ** (-1 / 4)) < 1e-9, '두 식이 같은 값')
ok(abs(ppl - 2 ** 1.75) < 1e-9, 'PPL = 2^1.75 = 약 3.364')

## 9. RNN 한 걸음

강의 N3 p.47, 실습 N3L p.10. h_t = tanh(W_h h_(t-1) + W_e e_t + b). 은닉 상태 2칸, 임베딩 2칸짜리 아주 작은 예예요.

In [ ]:
W_h = torch.tensor([[0.5, 0.0], [0.0, 0.5]])
W_e = torch.tensor([[1.0, 0.0], [0.0, -1.0]])
b = torch.tensor([0.0, 0.0])
h0 = torch.tensor([0.0, 0.0])
e1 = torch.tensor([1.0, 1.0])

h1 = None   # TODO: 위 식 그대로 (torch.tanh, @ 사용)
print(h1)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(torch.allclose(h1, torch.tensor([math.tanh(1.0), math.tanh(-1.0)])), 'h1 = [tanh(1), tanh(-1)] = [0.7616, -0.7616]')

## 10. 기울기 소실, 폭발, 클리핑

강의 N3 p.52-53. 1보다 작은 수를 여러 번 곱하면 0에 가까워지고(소실), 1보다 큰 수를 여러 번 곱하면 아주 커져요(폭발). **기울기 클리핑(Gradient Clipping)** 은 기울기 벡터의 길이가 기준보다 길면 방향은 두고 길이만 기준으로 줄여요.

In [ ]:
vanish = 0.5 ** 20
explode = 1.5 ** 20

g = [3.0, 4.0]
threshold = 1.0
length = math.sqrt(sum(v * v for v in g))
clipped = None   # TODO: length > threshold 이면 각 값에 threshold / length 를 곱한 리스트
print(vanish, round(explode, 1), clipped)

**확인 셀**: 실행해서 맞았는지 봐요.

In [ ]:
ok(vanish < 1e-6 and explode > 3000, '0.5^20 은 약 0.00000095, 1.5^20 은 약 3325')
ok(all(abs(a - b) < 1e-9 for a, b in zip(clipped, [0.6, 0.8])), '길이 5 를 1 로: [0.6, 0.8]')

## 정답 코드

먼저 스스로 풀어 보고, 막혔을 때만 봐요.

**1. 뉴런 하나 계산하기**

```python
x = [1.0, 2.0, -1.0]
w = [0.5, -0.25, 1.0]
b = 0.5

z = sum(xi * wi for xi, wi in zip(x, w)) + b
h = max(0.0, z)
print(z, h)
```

**2. 비선형이 없으면 층을 쌓아도 한 층**

```python
torch.manual_seed(0)
W1 = torch.randn(4, 3)
W2 = torch.randn(2, 4)
x = torch.randn(3)

two_layers = W2 @ (W1 @ x)
W = W2 @ W1
one_layer = W @ x
print(two_layers, one_layer, W.shape)
```

**3. shape 먼저 예측하기**

```python
x = torch.randn(5)
W = torch.randn(3, 5)
b = torch.randn(3)
u = torch.randn(3)

z = W @ x + b
s = u @ torch.tanh(z)

my_z_shape = (3,)
my_s_shape = ()
```

**4. 손으로 미분한 값과 autograd 비교**

```python
x = torch.tensor(1.0, requires_grad=True)
f = 3 * x**2 + 5 * x
f.backward()

by_hand = 6 * 1.0 + 5
print(x.grad.item(), by_hand)
```

**5. 연쇄 법칙**

```python
x = torch.tensor(1.0, requires_grad=True)
u = 2 * x + 1
f = u ** 2
f.backward()

df_du = 2 * 3.0
du_dx = 2.0
print(x.grad.item(), df_du * du_dx)
```

**6. 계산 그래프 노드 직관: +, max, x**

```python
grads_by_hand = {'x': 3.0, 'y': 3.0, 'z': 3.0, 'w': 0.0}

x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
z = torch.tensor(3.0, requires_grad=True)
w = torch.tensor(-1.0, requires_grad=True)
f = (x + y) * torch.maximum(z, w)
f.backward()
auto = {'x': x.grad.item(), 'y': y.grad.item(), 'z': z.grad.item(), 'w': w.grad.item()}
print(auto)
```

**7. 바이그램(2-gram) 언어 모델: 세어서 확률 구하기**

```python
from collections import Counter
sents = [['<s>', '나는', '영화', '를', '봤다', '</s>'],
         ['<s>', '나는', '책', '을', '봤다', '</s>'],
         ['<s>', '너는', '영화', '를', '봤다', '</s>']]

uni = Counter(w for s in sents for w in s[:-1])
bi = Counter((s[i], s[i + 1]) for s in sents for i in range(len(s) - 1))

def p(nxt, prev):
    return bi[(prev, nxt)] / uni[prev]

print(p('나는', '<s>'), p('영화', '나는'))
```

**8. 퍼플렉서티(Perplexity)**

```python
probs = [0.5, 0.25, 0.125, 0.5]

mean_nll = -sum(math.log(q) for q in probs) / len(probs)
ppl = math.exp(mean_nll)

print(round(mean_nll, 4), round(ppl, 4))
```

**9. RNN 한 걸음**

```python
W_h = torch.tensor([[0.5, 0.0], [0.0, 0.5]])
W_e = torch.tensor([[1.0, 0.0], [0.0, -1.0]])
b = torch.tensor([0.0, 0.0])
h0 = torch.tensor([0.0, 0.0])
e1 = torch.tensor([1.0, 1.0])

h1 = torch.tanh(W_h @ h0 + W_e @ e1 + b)
print(h1)
```

**10. 기울기 소실, 폭발, 클리핑**

```python
vanish = 0.5 ** 20
explode = 1.5 ** 20

g = [3.0, 4.0]
threshold = 1.0
length = math.sqrt(sum(v * v for v in g))
clipped = [v * threshold / length for v in g] if length > threshold else g
print(vanish, round(explode, 1), clipped)
```